 ## to be Understood

## DRG INSIGHTS SUMMARY

**Key Findings from Diagnosis Analysis:**

• **High-volume conditions** with largest total gaps: These drive absolute dollar impact
• **Ownership variation by DRG**: Some diagnoses hit Non-Profits harder than others
• **Geographic patterns**: Rural vs Metropolitan gaps vary significantly by diagnosis type  
• **Trend persistence**: Most gaps stable 2017-2019, disrupted by COVID in 2020
• **Severity matters**: Even after controlling for complexity, gaps remain significant

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "Data" / "Processed_Data"

df_medidata = pd.read_parquet(PROCESSED_DIR / "df_medidata.parquet")

print(f"Loaded data shape: {df_medidata.shape}")
print(f"Data ready for analysis")

Loaded data shape: (1178407, 20)
Data ready for analysis


In [2]:
print('\n' + '=' * 70)
print('SEVERITY-ADJUSTED PAYMENT GAPS')
print('=' * 70)
print('(Gap per discharge divided by DRG severity weight)')
print('=' * 70)

# Create severity-adjusted gap metric
df_medidata['gap_per_severity'] = (
    df_medidata['payment_gap_per_discharge'] / df_medidata['DRG_Weight']
)

top_drgs_severity = (
    df_medidata
    .groupby(['DRG_Cd', 'DRG_Desc'], as_index=False)
    .agg({
        'gap_per_severity': 'median',
        'Payment_Ratio': 'median',
        'DRG_Weight': 'median',
        'Tot_Dschrgs': 'sum'
    })
    .sort_values('gap_per_severity', ascending=False)
    .head(10)
)

print('\nTop 10 DRGs with Highest Severity-Adjusted Gap\n')
print('Gap/Severity | Payment_Ratio | DRG_Weight | Discharges | DRG Description')
print('-' * 70)
for idx, row in top_drgs_severity.iterrows():
    print(f"${row['gap_per_severity']:>10,.0f} | {row['Payment_Ratio']:>13.2%} | " + 
          f"{row['DRG_Weight']:>10.2f} | {row['Tot_Dschrgs']:>10,.0f} | {row['DRG_Desc'][:30]}")



SEVERITY-ADJUSTED PAYMENT GAPS
(Gap per discharge divided by DRG severity weight)


KeyError: 'payment_gap_per_discharge'

In [ ]:
print('\n' + '=' * 70)
print('PAYMENT GAP TRENDS BY YEAR (Top DRGs)')
print('=' * 70)

# Track top 3 DRGs over time
top_3_drgs = df_medidata.groupby('DRG_Cd').size().nlargest(3).index.tolist()

for drg in top_3_drgs:
    drg_name = df_medidata[df_medidata['DRG_Cd'] == drg]['DRG_Desc'].iloc[0]
    print(f"\nDRG {drg}: {drg_name[:50]}")
    
    trend = (
        df_medidata[df_medidata['DRG_Cd'] == drg]
        .groupby('Data_Year', as_index=False)
        .agg({
            'payment_gap_per_discharge': 'median',
            'Tot_Dschrgs': 'sum',
            'Payment_Ratio': 'median'
        })
        .sort_values('Data_Year')
    )
    
    for idx, row in trend.iterrows():
        print(f"  {int(row['Data_Year'])}: Gap/case ${row['payment_gap_per_discharge']:>8,.0f} | " + 
              f"Ratio {row['Payment_Ratio']:.2%} | {row['Tot_Dschrgs']:>8,.0f} cases")



PAYMENT GAP TRENDS BY YEAR (Top DRGs)

DRG 871: SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS W
  2017: Gap/case $     217 | Ratio 23.74% |  607,584 cases
  2018: Gap/case $     222 | Ratio 23.75% |  622,866 cases
  2019: Gap/case $     235 | Ratio 23.46% |  608,429 cases
  2020: Gap/case $     263 | Ratio 23.68% |  591,720 cases
  2021: Gap/case $     323 | Ratio 22.84% |  538,823 cases
  2022: Gap/case $     324 | Ratio 22.31% |  550,306 cases
  2023: Gap/case $     316 | Ratio 22.20% |  561,795 cases

DRG 291: HEART FAILURE AND SHOCK WITH MCC
  2017: Gap/case $     257 | Ratio 26.37% |  371,077 cases
  2018: Gap/case $     254 | Ratio 25.69% |  385,471 cases
  2019: Gap/case $     261 | Ratio 23.93% |  394,926 cases
  2020: Gap/case $     350 | Ratio 23.40% |  303,430 cases
  2021: Gap/case $     358 | Ratio 22.66% |  310,937 cases
  2022: Gap/case $     348 | Ratio 21.60% |  324,750 cases
  2023: Gap/case $     365 | Ratio 21.53% |  319,702 cases

DRG 193: SIMPLE PNEUMONIA AND 

In [ ]:
print('\n' + '=' * 70)
print('DRG PAYMENT GAPS BY OWNERSHIP TYPE')
print('=' * 70)

# Select top 10 DRGs by volume
top_10_by_volume = df_medidata.groupby('DRG_Cd').size().nlargest(10).index.tolist()
df_top_drgs = df_medidata[df_medidata['DRG_Cd'].isin(top_10_by_volume)].copy()

drg_ownership = (
    df_top_drgs
    .groupby(['DRG_Cd', 'DRG_Desc', 'Ownership_Type'], as_index=False)
    .agg({
        'Payment_Gap': 'mean',
        'payment_gap_per_discharge': 'median',
        'Tot_Dschrgs': 'sum'
    })
    .sort_values(['DRG_Cd', 'Payment_Gap'], ascending=[True, False])
)

for drg in top_10_by_volume[:5]:
    drg_name = df_medidata[df_medidata['DRG_Cd'] == drg]['DRG_Desc'].iloc[0]
    print(f"\nDRG {drg}: {drg_name[:50]}")
    subset = drg_ownership[drg_ownership['DRG_Cd'] == drg]
    for ownership in ['Government', 'Non-Profit', 'For-Profit']:
        row = subset[subset['Ownership_Type'] == ownership]
        if not row.empty:
            gap = row['payment_gap_per_discharge'].values[0]
            total_discharges = row['Tot_Dschrgs'].values[0]
            print(f"  {ownership:<15}: ${gap:>8,.0f}/case ({total_discharges:>8,.0f} discharges)")



DRG PAYMENT GAPS BY OWNERSHIP TYPE

DRG 871: SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS W
  Government     : $     288/case ( 376,352 discharges)
  Non-Profit     : $     496/case ( 711,592 discharges)
  For-Profit     : $     220/case (2,993,579 discharges)

DRG 291: HEART FAILURE AND SHOCK WITH MCC
  Government     : $     301/case ( 237,830 discharges)
  Non-Profit     : $     618/case ( 374,249 discharges)
  For-Profit     : $     255/case (1,798,214 discharges)

DRG 193: SIMPLE PNEUMONIA AND PLEURISY WITH MCC
  Government     : $     708/case (  92,772 discharges)
  Non-Profit     : $   1,408/case ( 148,026 discharges)
  For-Profit     : $     654/case ( 629,356 discharges)

DRG 872: SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS W
  Government     : $     609/case (  83,071 discharges)
  Non-Profit     : $   1,256/case ( 138,363 discharges)
  For-Profit     : $     543/case ( 672,438 discharges)

DRG 177: RESPIRATORY INFECTIONS AND INFLAMMATIONS WITH MCC
  Government    

In [ ]:
print('\n' + '=' * 70)
print('DRG PAYMENT GAPS BY GEOGRAPHY')
print('=' * 70)

drg_geography = (
    df_top_drgs
    .groupby(['DRG_Cd', 'DRG_Desc', 'RUCA_Group'], as_index=False)
    .agg({
        'payment_gap_per_discharge': 'median',
        'Tot_Dschrgs': 'sum'
    })
    .sort_values(['DRG_Cd', 'payment_gap_per_discharge'], ascending=[True, False])
)

ruca_order_local = ['Metropolitan', 'Micropolitan', 'Small Town', 'Rural']

for drg in top_10_by_volume[:3]:
    drg_name = df_medidata[df_medidata['DRG_Cd'] == drg]['DRG_Desc'].iloc[0]
    print(f"\nDRG {drg}: {drg_name[:50]}")
    subset = drg_geography[drg_geography['DRG_Cd'] == drg]
    for ruca in ruca_order_local:
        row = subset[subset['RUCA_Group'] == ruca]
        if not row.empty:
            gap = row['payment_gap_per_discharge'].values[0]
            total_discharges = row['Tot_Dschrgs'].values[0]
            print(f"  {ruca:<17}: ${gap:>8,.0f}/case ({total_discharges:>8,.0f} discharges)")



DRG PAYMENT GAPS BY GEOGRAPHY

DRG 871: SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS W
  Metropolitan     : $     252/case (3,538,307 discharges)
  Micropolitan     : $     290/case ( 437,044 discharges)
  Small Town       : $     375/case (  74,840 discharges)
  Rural            : $     314/case (  13,367 discharges)

DRG 291: HEART FAILURE AND SHOCK WITH MCC
  Metropolitan     : $     289/case (2,096,961 discharges)
  Micropolitan     : $     344/case ( 251,954 discharges)
  Small Town       : $     470/case (  41,832 discharges)
  Rural            : $     356/case (   8,756 discharges)

DRG 193: SIMPLE PNEUMONIA AND PLEURISY WITH MCC
  Metropolitan     : $     778/case ( 723,434 discharges)
  Micropolitan     : $     739/case ( 116,026 discharges)
  Small Town       : $     860/case (  21,931 discharges)
  Rural            : $     606/case (   4,348 discharges)


In [ ]:
print('=' * 70)
print('TOP 15 DRGs BY TOTAL PAYMENT GAP')
print('=' * 70)

top_drgs_gap = (
    df_medidata
    .groupby(['DRG_Cd', 'DRG_Desc'], as_index=False)
    .agg({
        'Payment_Gap': 'sum',
        'Tot_Dschrgs': 'sum',
        'DRG_Weight': 'median'
    })
    .sort_values('Payment_Gap', ascending=False)
    .head(15)
)

top_drgs_gap['Gap_Per_Discharge'] = top_drgs_gap['Payment_Gap'] / top_drgs_gap['Tot_Dschrgs']
top_drgs_gap['DRG_Label'] = top_drgs_gap['DRG_Cd'].astype(str) + ' - ' + top_drgs_gap['DRG_Desc'].str[:40]

print('\nTotal Gap | Gap/Case | Discharges | Severity (DRG_Weight)')
print('-' * 70)
for idx, row in top_drgs_gap.iterrows():
    print(f"${row['Payment_Gap']/1e9:.2f}B | ${row['Gap_Per_Discharge']:,.0f} | {row['Tot_Dschrgs']:,.0f} | {row['DRG_Weight']:.2f}")
    print(f"  {row['DRG_Label']}\n")


TOP 15 DRGs BY TOTAL PAYMENT GAP

Total Gap | Gap/Case | Discharges | Severity (DRG_Weight)
----------------------------------------------------------------------
$1.85B | $3,228 | 573,863 | 5.06
  853 - INFECTIOUS AND PARASITIC DISEASES WITH O

$1.59B | $7,936 | 200,803 | 6.32
  870 - SEPTICEMIA OR SEVERE SEPSIS WITH MV >96 

$1.46B | $23,052 | 63,273 | 18.95
  3 - ECMO OR TRACHEOSTOMY WITH MV >96 HOURS O

$1.07B | $5,893 | 182,176 | 4.91
  329 - MAJOR SMALL AND LARGE BOWEL PROCEDURES W

$1.03B | $3,594 | 287,873 | 3.17
  246 - PERCUTANEOUS CARDIOVASCULAR PROCEDURES W

$0.98B | $240 | 4,081,523 | 1.87
  871 - SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >

$0.97B | $20,591 | 47,130 | 11.54
  4 - TRACHEOSTOMY WITH MV >96 HOURS OR PRINCI

$0.93B | $9,354 | 98,918 | 5.73
  207 - RESPIRATORY SYSTEM DIAGNOSIS WITH VENTIL

$0.89B | $1,843 | 480,993 | 2.03
  247 - PERCUTANEOUS CARDIOVASCULAR PROCEDURES W

$0.86B | $467 | 1,841,326 | 1.99
  470 - MAJOR HIP AND KNEE JOINT REPLACEMENT OR 

$0.86B | $

## 5. DIAGNOSIS-LEVEL ANALYSIS (DRG)

What diagnoses drive the largest payment gaps?